# 05 — Base Model Baseline on benchmark_v1

**Task 09/08 deliverable** — Runnable baseline notebook (skeleton).

**Status:** Skeleton — actual baseline run is deferred to Colab on 10/08 (per Q2 = option a).
The VPS provided by EVVO Labs is CPU-only and must not be used for inference/training.

This notebook:
1. Loads `data/benchmark/benchmark_v1.jsonl` (34 cases, 8 task types).
2. Defines a base-model loader stub (`load_base_model`) that returns `None` locally and a real HuggingFace pipeline on Colab.
3. Defines per-task-type metric functions that compare model output to `gold_output`.
4. Defines an error categorization function that buckets failures using the 13 codes from `data/benchmark/error_categories.md`.
5. Runs the baseline loop (skipped locally; runs end-to-end on Colab).
6. Writes `data/benchmark/baseline_result.json`.

**Re-run on Colab:**
1. Upload the repo (or clone from GitHub).
2. `pip install transformers torch accelerate jsonschema`.
3. Set `RUN_BASELINE = True` and `MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"` (or `1.5B` / `7B`).
4. Run All.

**Locally (CPU-only VPS):**
- Set `RUN_BASELINE = False` (default).
- Notebook produces a placeholder `baseline_result.json` with all-zero metrics, ready to be overwritten on Colab.


In [1]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from collections import Counter
from typing import Any, Optional

# ── Configuration ────────────────────────────────────────────────────
# Set RUN_BASELINE = True ONLY on Colab (or any GPU-equipped machine).
# The EVVO VPS is CPU-only — running inference there is forbidden.
RUN_BASELINE = False

# HuggingFace model ID for the base SLM. Candidate models (decided 10/08):
#   - Qwen/Qwen2.5-0.5B-Instruct  (smallest, fastest, lowest quality)
#   - Qwen/Qwen2.5-1.5B-Instruct  (balanced)
#   - Qwen/Qwen2.5-7B-Instruct    (best quality, needs ~16GB VRAM in 4-bit)
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# ── Repo paths ──────────────────────────────────────────────────────
# Auto-resolve repo root from notebook location when running on Colab
# (works after `git clone` or after uploading the repo folder).
try:
    REPO_ROOT = Path.cwd()
    # Walk up until we find the data/benchmark folder
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "data" / "benchmark").exists():
        REPO_ROOT = REPO_ROOT.parent
except Exception:
    REPO_ROOT = Path(r"D:\evvo-slm-harness")

BENCHMARK_PATH = REPO_ROOT / "data" / "benchmark" / "benchmark_v1.jsonl"
ERROR_CATEGORIES_PATH = REPO_ROOT / "data" / "benchmark" / "error_categories.md"
BASELINE_RESULT_PATH = REPO_ROOT / "data" / "benchmark" / "baseline_result.json"

print(f"RUN_BASELINE = {RUN_BASELINE}")
print(f"MODEL_ID     = {MODEL_ID}")
print(f"REPO_ROOT    = {REPO_ROOT}")
print(f"Benchmark    = {BENCHMARK_PATH}")
print(f"Exists?      = {BENCHMARK_PATH.exists()}")


RUN_BASELINE = False
MODEL_ID     = Qwen/Qwen2.5-0.5B-Instruct
REPO_ROOT    = D:\evvo-slm-harness
Benchmark    = D:\evvo-slm-harness\data\benchmark\benchmark_v1.jsonl
Exists?      = True


## 1. Load benchmark

Load all 34 cases from `benchmark_v1.jsonl` and print a coverage summary.


In [2]:
def load_benchmark(path: Path) -> list[dict]:
    """Load all cases from a JSONL benchmark file."""
    cases = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                cases.append(json.loads(line))
    return cases

cases = load_benchmark(BENCHMARK_PATH)
print(f"Loaded {len(cases)} cases from {BENCHMARK_PATH.name}")
print()

by_task = Counter(c["task_type"] for c in cases)
by_diff = Counter(c["metadata"]["difficulty"] for c in cases)
hard_neg = sum(1 for c in cases if c["metadata"].get("is_hard_negative"))

print("By task type:")
for tt in sorted(by_task):
    print(f"  {tt:35} {by_task[tt]:3d}")
print(f"\nBy difficulty: {dict(sorted(by_diff.items()))}")
print(f"Hard negatives: {hard_neg}")


Loaded 34 cases from benchmark_v1.jsonl

By task type:
  client_qa                             7
  evidence_check                        5
  false_positive_detection              3
  finding_review                        5
  hard_negative_potential_issue         1
  remediation_review                    5
  severity_review                       5
  unsupported_refusal                   3

By difficulty: {'easy': 14, 'hard': 9, 'medium': 11}
Hard negatives: 9


## 2. Base model loader

Stub function that returns `None` locally and a real HuggingFace pipeline on Colab.

The actual model selection (between Qwen2.5-0.5B / 1.5B / 7B-Instruct) is decided on 10/08 based on Colab GPU tier (free T4 vs Pro A100) and VRAM headroom.


In [3]:
def load_base_model(model_id: str):
    """Load a base SLM for inference.

    Returns a callable `generate(prompt: str) -> str` on Colab, or None locally.

    On Colab, this function:
      1. Imports torch + transformers.
      2. Loads the model in 4-bit (BitsAndBytes) for 7B, or fp16 for 0.5B/1.5B.
      3. Wraps it in a `pipeline("text-generation", ...)`.
      4. Returns a closure that formats the prompt as a chat and returns the generated text.
    """
    if not RUN_BASELINE:
        print(f"[stub] RUN_BASELINE=False — would load {model_id} on Colab.")
        return None

    import torch
    from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

    print(f"Loading {model_id} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Use 4-bit quantization for 7B models, fp16 for smaller ones.
    if "7B" in model_id:
        from transformers import BitsAndBytesConfig

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb_config, device_map="auto",
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.float16, device_map="auto",
        )

    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512)

    def generate(prompt: str) -> str:
        messages = [{"role": "user", "content": prompt}]
        out = pipe(messages, return_full_text=False)[0]["generated_text"]
        return out

    return generate

model = load_base_model(MODEL_ID)
print(f"model loaded: {model is not None}")


[stub] RUN_BASELINE=False — would load Qwen/Qwen2.5-0.5B-Instruct on Colab.
model loaded: False


## 3. Prompt builder

Convert a benchmark case into a single prompt string for the SLM.

The prompt includes:
- The instruction.
- The input (JSON-serialized).
- A directive to return strict JSON matching the output schema.


In [4]:
def build_prompt(case: dict) -> str:
    """Render a benchmark case as a chat prompt for the base SLM."""
    instruction = case["instruction"]
    task_type = case["task_type"]
    payload = json.dumps(case["input"], ensure_ascii=False, indent=2)
    return (
        f"You are a pentest report reviewer. Complete the following task.\n\n"
        f"Task type: {task_type}\n"
        f"Instruction: {instruction}\n\n"
        f"Input:\n{payload}\n\n"
        f"Return ONLY valid JSON matching the output schema for this task type. "
        f"Do not include markdown fences or commentary."
    )

# Sanity check: render the first case
sample_prompt = build_prompt(cases[0])
print(f"Prompt length: {len(sample_prompt)} chars")
print("First 500 chars:")
print(sample_prompt[:500])
print("...")
print("Last 200 chars:")
print(sample_prompt[-200:])


Prompt length: 2463 chars
First 500 chars:
You are a pentest report reviewer. Complete the following task.

Task type: evidence_check
Instruction: Evaluate whether the exploitation evidence sufficiently supports the vulnerability claim. List unsupported claims and missing evidence.

Input:
{
  "finding_id": "FND-000001",
  "title": "Hardcoded RabbitMQ Credentials in Mobile Application",
  "severity": "critical",
  "observation": "The Android mobile application contains hardcoded RabbitMQ connection credentials embedded within the applica
...
Last 200 chars:
e as shown as in the screenshot:",
      "availability": "full_content"
    }
  ]
}

Return ONLY valid JSON matching the output schema for this task type. Do not include markdown fences or commentary.


## 4. Output parser

Parse the model's raw text output into a dict. Tolerates:
- Plain JSON.
- JSON wrapped in markdown fences (` ```json ... ``` `).
- JSON with leading/trailing commentary.

Returns `(parsed_dict | None, error_code | None)`.


In [5]:
import re

FENCE_RE = re.compile(r"```(?:json)?\s*(.*?)\s*```", re.DOTALL)

def parse_output(raw: str) -> tuple[Optional[dict], Optional[str]]:
    """Parse model output into a dict. Returns (parsed, error_code)."""
    if not raw or not raw.strip():
        return None, "FMT-INVALID-JSON"

    text = raw.strip()

    # Try markdown fence extraction first
    m = FENCE_RE.search(text)
    if m:
        text = m.group(1).strip()

    # Try to locate the outermost JSON object
    if not text.startswith("{"):
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return None, "FMT-INVALID-JSON"
        text = text[start:end+1]

    try:
        parsed = json.loads(text)
        if not isinstance(parsed, dict):
            return None, "FMT-INVALID-JSON"
        return parsed, None
    except json.JSONDecodeError:
        return None, "FMT-INVALID-JSON"

# Sanity check
test1 = "```json\n{\"a\": 1}\n```"
test2 = "Here is the answer: {\"a\": 1} thanks"
test3 = "not json at all"
for t in [test1, test2, test3]:
    parsed, err = parse_output(t)
    print(f"{t[:30]!r:35} -> parsed={parsed}, err={err}")


'```json\n{"a": 1}\n```'            -> parsed={'a': 1}, err=None
'Here is the answer: {"a": 1} t'    -> parsed={'a': 1}, err=None
'not json at all'                   -> parsed=None, err=FMT-INVALID-JSON


## 5. Per-task metric functions

Each function takes `(predicted: dict, gold: dict)` and returns a list of error codes (empty if pass).


In [6]:
def _get(obj: dict, *path, default=None):
    cur = obj
    for k in path:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

def metric_classification(predicted: dict, gold: dict, task_type: str) -> list[str]:
    errors = []
    gold_label = _get(gold, "classification", "label")
    pred_label = _get(predicted, "classification", "label")
    if pred_label is None:
        errors.append("FMT-MISSING-FIELD")
    elif pred_label != gold_label:
        errors.append("CLASS-WRONG-LABEL")
        if pred_label == "confirmed_vulnerability" and gold_label in ("potential_issue", "false_positive"):
            errors.append("CLASS-HALLUCINATED-CONFIRM")
    return errors

def metric_evidence(predicted: dict, gold: dict) -> list[str]:
    errors = []
    gold_suff = _get(gold, "evidence_review", "is_sufficient")
    pred_suff = _get(predicted, "evidence_review", "is_sufficient")
    if pred_suff is None:
        errors.append("FMT-MISSING-FIELD")
    elif pred_suff != gold_suff:
        errors.append("EVID-WRONG-SUFFICIENCY")
        if pred_suff is True and gold_suff is False:
            errors.append("EVID-HALLUCINATED")
    gold_unsupp = _get(gold, "evidence_review", "unsupported_claims", default=[]) or []
    pred_unsupp = _get(predicted, "evidence_review", "unsupported_claims", default=[]) or []
    if gold_unsupp and not pred_unsupp:
        errors.append("EVID-MISSED-UNSUPPORTED")
    return errors

def metric_severity(predicted: dict, gold: dict) -> list[str]:
    errors = []
    gold_sugg = _get(gold, "severity_review", "suggested_severity")
    pred_sugg = _get(predicted, "severity_review", "suggested_severity")
    if pred_sugg is None:
        errors.append("FMT-MISSING-FIELD")
    elif pred_sugg != gold_sugg:
        errors.append("SEV-WRONG-SUGGESTION")
    gold_change = _get(gold, "severity_review", "change_recommended")
    pred_change = _get(predicted, "severity_review", "change_recommended")
    if gold_change is True and pred_change is False:
        errors.append("SEV-MISSED-MISMATCH")
    return errors

def metric_qa(predicted: dict, gold: dict) -> list[str]:
    errors = []
    gold_refuses = _get(gold, "refuses", default=False)
    pred_refuses = _get(predicted, "refuses", default=None)
    if pred_refuses is None:
        errors.append("FMT-MISSING-FIELD")
    elif gold_refuses and not pred_refuses:
        errors.append("CLASS-FAILED-REFUSE")
    elif not gold_refuses and pred_refuses:
        errors.append("QA-REFUSED-IN-SCOPE")
    # Redaction leak check
    pred_answer = _get(predicted, "answer", default="") or ""
    if "[REDACTED_" in str(pred_answer) or "password" in str(pred_answer).lower():
        if gold_refuses:
            errors.append("QA-LEAKED-REDACTED")
    return errors

def score_case(predicted: dict | None, gold: dict, task_type: str, parse_error: str | None) -> list[str]:
    """Return list of error codes for a single case."""
    if parse_error:
        return [parse_error]
    if predicted is None:
        return ["FMT-INVALID-JSON"]

    errors: list[str] = []
    if task_type in ("finding_review", "hard_negative_potential_issue", "false_positive_detection"):
        errors.extend(metric_classification(predicted, gold, task_type))
    if task_type in ("finding_review", "evidence_check", "hard_negative_potential_issue"):
        errors.extend(metric_evidence(predicted, gold))
    if task_type in ("finding_review", "severity_review"):
        errors.extend(metric_severity(predicted, gold))
    if task_type in ("client_qa", "unsupported_refusal"):
        errors.extend(metric_qa(predicted, gold))
    # remediation_review has no numeric metric in v0.1 — defer to 17/08 full taxonomy
    return errors

print("Metric functions defined.")


Metric functions defined.


## 6. Baseline run loop

Iterates over all 34 cases, calls the model, parses output, scores.

When `RUN_BASELINE = False`, the loop is skipped and a placeholder result is produced.


In [7]:
def run_baseline(model, cases: list[dict]) -> list[dict]:
    """Run the model over all cases. Returns per_case results."""
    per_case = []
    for i, case in enumerate(cases, start=1):
        case_id = case["case_id"]
        task_type = case["task_type"]
        gold = case["gold_output"]

        if model is None:
            per_case.append({
                "case_id": case_id,
                "task_type": task_type,
                "errors": ["FMT-INVALID-JSON"],  # placeholder — no model run
                "latency_ms": 0,
                "raw_output": None,
            })
            continue

        prompt = build_prompt(case)
        t0 = datetime.now(timezone.utc)
        try:
            raw = model(prompt)
        except Exception as e:
            raw = ""
            print(f"  [{i}/{len(cases)}] {case_id}: model error: {e}")
        t1 = datetime.now(timezone.utc)
        latency_ms = int((t1 - t0).total_seconds() * 1000)

        parsed, parse_err = parse_output(raw)
        errors = score_case(parsed, gold, task_type, parse_err)

        per_case.append({
            "case_id": case_id,
            "task_type": task_type,
            "errors": errors,
            "latency_ms": latency_ms,
            "raw_output": raw,
        })

        status = "PASS" if not errors else "FAIL: " + ",".join(errors)
        print(f"  [{i}/{len(cases)}] {case_id} ({task_type:30}) {latency_ms:5}ms  {status}")

    return per_case

print("Running baseline...")
per_case = run_baseline(model, cases)
print(f"\nDone. {len(per_case)} cases scored.")


Running baseline...

Done. 34 cases scored.


## 7. Aggregate and write baseline_result.json

Compute pass rate, error distribution, hard-negative pass rate, and write to `data/benchmark/baseline_result.json`.


In [8]:
def aggregate(per_case: list[dict], cases: list[dict]) -> dict:
    case_by_id = {c["case_id"]: c for c in cases}

    n_total = len(per_case)
    n_pass = sum(1 for pc in per_case if not pc["errors"])
    n_errors = sum(len(pc["errors"]) for pc in per_case)

    hard_neg_ids = {c["case_id"] for c in cases if c["metadata"].get("is_hard_negative")}
    hard_neg_per_case = [pc for pc in per_case if pc["case_id"] in hard_neg_ids]
    hard_neg_pass = sum(1 for pc in hard_neg_per_case if not pc["errors"])

    error_dist = Counter()
    for pc in per_case:
        for e in pc["errors"]:
            error_dist[e] += 1

    # Ensure all 13 error codes are present (fill 0 for missing)
    ALL_CODES = [
        "CLASS-WRONG-LABEL", "CLASS-HALLUCINATED-CONFIRM", "CLASS-FAILED-REFUSE",
        "EVID-WRONG-SUFFICIENCY", "EVID-HALLUCINATED", "EVID-MISSED-UNSUPPORTED",
        "SEV-WRONG-SUGGESTION", "SEV-MISSED-MISMATCH",
        "QA-LEAKED-REDACTED", "QA-REFUSED-IN-SCOPE",
        "FMT-INVALID-JSON", "FMT-SCHEMA-VIOLATION", "FMT-MISSING-FIELD",
    ]
    error_distribution = {code: error_dist.get(code, 0) for code in ALL_CODES}

    return {
        "benchmark_version": "1.0",
        "model_id": MODEL_ID if RUN_BASELINE else None,
        "ran_at": datetime.now(timezone.utc).isoformat() if RUN_BASELINE else None,
        "environment": "google_colab" if RUN_BASELINE else "local_stub",
        "case_count": n_total,
        "pass_rate": round(n_pass / n_total, 4) if n_total else 0.0,
        "error_rate": round(n_errors / n_total, 4) if n_total else 0.0,
        "hard_negative_pass_rate": (
            round(hard_neg_pass / len(hard_neg_ids), 4) if hard_neg_ids else 0.0
        ),
        "error_distribution": error_distribution,
        "per_case": [
            {k: v for k, v in pc.items() if k != "raw_output"}
            for pc in per_case
        ],
        "notes": [
            "Placeholder structure. Fill in by running notebooks/05_baseline.ipynb on Colab (10/08 task).",
            "All values are null/zero until the baseline is actually run.",
            "Per-case entries follow the shape documented in data/benchmark/error_categories.md section 6.",
        ] if not RUN_BASELINE else [
            f"Baseline run on {MODEL_ID} via Colab.",
            "Gold labels are rule-based — see data/benchmark/data_card.md section 4.3.",
        ],
    }

result = aggregate(per_case, cases)
print(f"pass_rate:              {result['pass_rate']}")
print(f"error_rate:             {result['error_rate']}")
print(f"hard_negative_pass_rate:{result['hard_negative_pass_rate']}")
print("error_distribution:")
for code, count in result["error_distribution"].items():
    if count:
        print(f"  {code:35} {count}")

# Write to disk
BASELINE_RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
with BASELINE_RESULT_PATH.open("w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)
print(f"\nWrote {BASELINE_RESULT_PATH}")


pass_rate:              0.0
error_rate:             1.0
hard_negative_pass_rate:0.0
error_distribution:
  FMT-INVALID-JSON                    34

Wrote D:\evvo-slm-harness\data\benchmark\baseline_result.json


## 8. Per-task-type breakdown

Show pass rate per task type — useful for spotting which task types the base model struggles with.


In [9]:
from collections import defaultdict

per_task_stats = defaultdict(lambda: {"total": 0, "pass": 0, "errors": []})
for pc in per_case:
    tt = pc["task_type"]
    per_task_stats[tt]["total"] += 1
    if not pc["errors"]:
        per_task_stats[tt]["pass"] += 1
    per_task_stats[tt]["errors"].extend(pc["errors"])

print(f"{'task_type':35} {'pass/total':12} {'pass_rate':10} top_errors")
print("-" * 90)
for tt in sorted(per_task_stats):
    s = per_task_stats[tt]
    rate = s["pass"] / s["total"] if s["total"] else 0
    err_counter = Counter(s["errors"])
    top = ", ".join(f"{c}({n})" for c, n in err_counter.most_common(3))
    print(f"{tt:35} {s['pass']:>4}/{s['total']:<7} {rate:>8.1%}  {top}")


task_type                           pass/total   pass_rate  top_errors
------------------------------------------------------------------------------------------
client_qa                              0/7           0.0%  FMT-INVALID-JSON(7)
evidence_check                         0/5           0.0%  FMT-INVALID-JSON(5)
false_positive_detection               0/3           0.0%  FMT-INVALID-JSON(3)
finding_review                         0/5           0.0%  FMT-INVALID-JSON(5)
hard_negative_potential_issue          0/1           0.0%  FMT-INVALID-JSON(1)
remediation_review                     0/5           0.0%  FMT-INVALID-JSON(5)
severity_review                        0/5           0.0%  FMT-INVALID-JSON(5)
unsupported_refusal                    0/3           0.0%  FMT-INVALID-JSON(3)


## 9. Next steps

**If RUN_BASELINE = False (local stub):**
- The `baseline_result.json` written above is a placeholder (all-zero metrics).
- To get real numbers, upload this notebook to Colab, set `RUN_BASELINE = True`, and Run All.

**If RUN_BASELINE = True (Colab):**
- Compare `pass_rate` against the fine-tuned SLM v0.1 baseline (week 11 task).
- Use the per-task-type breakdown to identify which task types need more training data.
- Use the error distribution to prioritize the error taxonomy for 17/08.

**Comparison matrix (planned for 16/08):**

| Configuration        | pass_rate | hard_neg_pass_rate | top error code |
|----------------------|-----------|--------------------|----------------|
| Base                 |    ?      |       ?            |       ?        |
| Base + RAG           |    ?      |       ?            |       ?        |
| Fine-tuned           |    ?      |       ?            |       ?        |
| Fine-tuned + RAG     |    ?      |       ?            |       ?        |
